# UC3 — PG + private ES + dual-search + privacy probe

**Persona:** publisher of restricted-access vector data.

**Goal:**
1. Configure WRITE to PG (primary) + private ES (async) — no public ES.
2. Configure SEARCH/READ to private ES (`geometry_simplified`) + PG
   (`geometry_exact`).
3. Verify that **anonymous** SEARCH cannot leak items.  Authenticated
   SEARCH returns features via either dispatch path.

Privacy is expressed by **routing-pin presence** of
`items_elasticsearch_private_driver` (or its collection counterpart) —
pinning a private driver in `ItemsRoutingConfig` is what makes the
collection private.  The cascade enforcer rejects mixed public+private
driver pins in the same routing config (#733 retired the standalone
`CollectionPrivacy.is_private` flag in favour of this routing-driven
model).

In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"cf_uc3_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
if not TOKEN:
    print("\nWARNING: no Bearer token set — config writes will 401.")
    print("Set DYNASTORE_TOKEN before running write cells.")


In [ ]:
# Create catalog (idempotent: 409 = already exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"Cycle F UC walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for cycle_f_use_cases notebook.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
if r.status_code in (200, 201):
    print(f"Catalog '{CATALOG_ID}' created.")
elif r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Create collection (idempotent)
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UC collection {RUN_ID}",
    "description": "Walkthrough collection — defaults inherited then PATCHed.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


In [ ]:
# Helper — show explicit + waterfall-resolved view for a plugin.
#
# The configs API exposes:
#   GET /configs/.../plugins/{plugin_id}                  → explicit (per-scope)
#   GET /configs/.../plugins/{plugin_id}?resolved=true    → waterfall-resolved
# There is NO `/effective` endpoint — the resolved form is reached via
# the `?resolved=true` query string.
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        base = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    elif level == "catalog":
        base = f"/configs/catalogs/{CATALOG_ID}"
    else:
        raise ValueError(level)
    rx = client.get(f"{base}/plugins/{plugin_id}")
    rr = client.get(f"{base}/plugins/{plugin_id}", params={"resolved": "true"})
    print(f"\n=== {plugin_id} @ {level} ===")
    print("EXPLICIT (only fields stored at this scope):")
    if rx.status_code == 200:
        print(json.dumps(rx.json(), indent=2)[:600])
    else:
        print(f"  ({rx.status_code} — none stored, every field inherited)")
    print("RESOLVED (waterfall over platform → catalog → collection):")
    if rr.status_code == 200:
        print(json.dumps(rr.json(), indent=2)[:800])
    else:
        print(f"  ({rr.status_code}) {rr.text[:160]}")


In [ ]:
def put_config(plugin_id: str, body: dict, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    else:
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    r = client.put(url, content=json.dumps(body), timeout=60.0)
    print(f"PUT {plugin_id}: {r.status_code}")
    if r.status_code not in (200, 201, 204):
        print(f"  body: {r.text[:300]}")
        if r.status_code == 401:
            raise RuntimeError("Unauthorized — set DYNASTORE_TOKEN.")
        raise RuntimeError(f"PUT failed: {r.status_code}")


## Step 1 — PATCH PG driver (sidecars: geometries + attributes)

In [ ]:
items_pg_driver = {
    "engine_ref": "postgresql_engine_config",
    "sidecars": [
        {"sidecar_type": "geometries"},
        {
            "sidecar_type": "attributes",
            # `external_id_field` lives on items_write_policy, not on the
            # sidecar — sidecar config carries only DDL flags.
            "enable_external_id": True,
            "enable_asset_id": True,
        },
    ],
}
put_config("items_postgresql_driver_config", items_pg_driver)


## Step 2 — PATCH routing for PG + private ES (no public ES)

In [ ]:
routing = {
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver",         "on_failure": "fatal"},
            {"driver_ref": "items_elasticsearch_private_driver", "write_mode": "async", "on_failure": "outbox"},
        ],
        "READ": [
            {"driver_ref": "items_elasticsearch_private_driver", "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",         "hints": ["geometry_exact"],     "on_failure": "fatal"},
        ],
        "SEARCH": [
            {"driver_ref": "items_elasticsearch_private_driver", "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",         "hints": ["geometry_exact"],     "on_failure": "fatal"},
        ],
    },
}
put_config("items_routing_config", routing)
show_config_delta("items_routing_config")


## Step 3 — Confirm routing config validation rejects mixed driver pins

Attempting to add `items_elasticsearch_driver` (public) to a routing
config that already pins `items_elasticsearch_private_driver` must fail
with 4xx — the routing config validator rejects entries with conflicting
driver pins for the same operation.

In [ ]:
bad_routing = json.loads(json.dumps(routing))  # deep copy
bad_routing["operations"]["WRITE"].append({
    "driver_ref": "items_elasticsearch_driver",  # public — should be rejected
    "write_mode": "async", "on_failure": "outbox",
})
url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/items_routing_config"
r = client.put(url, content=json.dumps(bad_routing))
print(f"PUT routing with public ES on private collection: {r.status_code}")
print(f"  body: {r.text[:300]}")
assert r.status_code in (422, 409, 400), \
    f"Expected 422/409/400 from privacy cascade, got {r.status_code}"


## Step 4 — Ingest 3 features

In [ ]:
ingest_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
for i, code_val in enumerate(["P-001", "P-002", "P-003"]):
    feat = {
        "type": "Feature",
        "stac_version": "1.0.0",
        "id": f"priv-{i}-{RUN_ID}",
        "collection": COLLECTION_ID,
        "geometry": {"type": "Point", "coordinates": [12.5 + i*0.05, 41.9]},
        "bbox": [12.5 + i*0.05, 41.9, 12.5 + i*0.05, 41.9],
        "properties": {"datetime": "2024-01-10T00:00:00Z", "code": code_val},
        "assets": {},
        "links": [],
    }
    r = client.post(ingest_url, content=json.dumps(feat))
    print(f"  ingest {code_val}: {r.status_code}")


## Step 5 — Anonymous probe must NOT leak items

Open a fresh client without the Bearer.  SEARCH should return 401/403
(or an empty list) — never the actual rows.


In [ ]:
anon = httpx.Client(base_url=BASE_URL, timeout=60.0)  # no auth
search_url = f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"
r = anon.get(search_url, params={"limit": 5})
print(f"anonymous SEARCH: {r.status_code}")
print(f"  body: {r.text[:200]}")
assert r.status_code in (401, 403, 404) or len(r.json().get("features", [])) == 0, \
    f"PRIVACY LEAK: anonymous returned items! {r.text[:200]}"
anon.close()


## Step 6 — Authenticated SEARCH returns the items

Hint dispatch (private ES vs PG) is internal — the search service
derives `filter_hints` from the query, then `router.resolve_drivers`
picks the driver whose `supported_hints` intersect.  The user-facing
surface is just "give me the items"; the routing is invisible.


In [ ]:
if not TOKEN:
    print("(skipped: no token)")
else:
    r = client.get(search_url, params={"limit": 5})
    feats = r.json().get("features", []) if r.status_code == 200 else []
    print(f"  authenticated SEARCH: HTTP {r.status_code} count={len(feats)}")
    for f in feats[:3]:
        print(f"    {f.get('id')} code={f.get('properties',{}).get('code')!r}")


## Teardown

In [ ]:
# Teardown — delete the ephemeral catalog. Comment out to keep state.
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
    timeout=60.0,
)
print(f"teardown DELETE /stac/catalogs/{CATALOG_ID}: {r.status_code}")
client.close()
